# Lesson 01: Shared foundations (solution)

**Goal:** understand the immutable data types and the hashing that every stage builds on.

**Before you start**
- Theory: [lesson 01](../../../docs/lessons/01_shared_foundations.md)
- Python concepts: Pydantic models, `frozen=True`, `model_validator`, `Self`, `try/except`
- Files used: `src/docrag/shared/schema.py`, `src/docrag/shared/cache.py`, `src/docrag/shared/config.py`. Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
from typing import Self

from pydantic import BaseModel, ConfigDict, ValidationError, model_validator

from docrag.shared.cache import stable_hash
from docrag.shared.config import ExperimentConfig
from docrag.shared.schema import Block

block = Block(doc_id="d", type="text", text="hello", page=1, section_path=[], order=0)
print(block)

## Step 1: Frozen models refuse changes

`Block` is declared with `model_config = ConfigDict(frozen=True)`: like a Java `record` or a Rust struct without `mut`. Assigning to a field raises a `ValidationError`. Catch it with `try/except` (Python's `try/catch`).

In [ ]:
caught = None
try:
    block.text = "changed"
except ValidationError as err:
    caught = err

In [ ]:
assert caught is not None, (
    "Hint: try: block.text = ...  except ValidationError as err: caught = err"
)
assert block.text == "hello", "The block must be unchanged"
print(type(caught).__name__, "->", caught.errors()[0]["type"])
print("OK")

## Step 2: Your own model with a validator

Write `PageSpan(page_start, page_end)`: frozen, and invalid when `page_end < page_start`. A `@model_validator(mode="after")` runs once all fields are set (a post-construct hook); raising `ValueError` inside it becomes a `ValidationError`. It must `return self` (typed `Self`).

In [ ]:
class PageSpan(BaseModel):
    model_config = ConfigDict(frozen=True)
    page_start: int
    page_end: int

    @model_validator(mode="after")
    def _end_after_start(self) -> Self:
        if self.page_end < self.page_start:
            raise ValueError("page_end must be >= page_start")
        return self

In [ ]:
span = PageSpan(page_start=2, page_end=3)
try:
    PageSpan(page_start=3, page_end=1)
    raise AssertionError("PageSpan(3, 1) should fail: add the model_validator")
except ValidationError:
    pass
try:
    span.page_end = 9
    raise AssertionError("Assignment should fail: make the model frozen")
except ValidationError:
    pass
assert len({span, PageSpan(page_start=2, page_end=3)}) == 1, (
    "Frozen models are hashable: equal spans collapse in a set"
)
print("OK")

## Step 3: Stable hashes for cache keys

`stable_hash(*parts)` is SHA-256 over canonical JSON (`sort_keys=True`), so the same content gives the same key on every machine and run. That's what makes the parse/embedding/LLM caches safe. Compute the hash of two dicts that differ only in key order.

In [ ]:
h1 = stable_hash({"a": 1, "b": 2})
h2 = stable_hash({"b": 2, "a": 1})

In [ ]:
assert h1 is not None and h2 is not None, "Hint: h1 = stable_hash({...})"
assert h1 == h2 and len(h1) == 64, "Key order must not matter; SHA-256 hex has 64 chars"
# The same idea identifies experiments: `name` is only a label, so it's excluded.
assert (
    ExperimentConfig(name="a").config_hash() == ExperimentConfig(name="b").config_hash()
)
assert (
    ExperimentConfig(name="a").config_hash()
    != ExperimentConfig(name="a", top_k=5).config_hash()
)
print(h1[:16], "...")
print("OK")

## Observation question

Why does assigning to a frozen field raise, and why is that worth it here?

<details>
<summary>Answer</summary>

`frozen=True` makes Pydantic's `__setattr__` raise instead of changing the field. Blocks and Chunks travel through every stage (parser, chunker, store, Verify) and across processes (the parse workers), and they are cached. If any stage could mutate them, a bug in one stage would silently change data another stage already used, and cached values could stop matching their keys. Immutability also makes them hashable, so they can be set members and dict keys.
</details>

## The real code

- Data contracts: [`src/docrag/shared/schema.py`](../../../src/docrag/shared/schema.py)
- Cache + stable_hash: [`src/docrag/shared/cache.py`](../../../src/docrag/shared/cache.py)
- ExperimentConfig: [`src/docrag/shared/config.py`](../../../src/docrag/shared/config.py)
- Lesson: [`docs/lessons/01_shared_foundations.md`](../../../docs/lessons/01_shared_foundations.md)